# Required imports
Always run

In [1]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import os

2026-04-18 13:28:51.386680: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-18 13:28:51.441869: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-18 13:28:52.805154: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Load feature

In [2]:
# Carica features da disco
X_train = np.load("features/X_train.npy")
X_test = np.load("features/X_test.npy")
y_train = np.load("features/y_train.npy")
y_test = np.load("features/y_test.npy")
labels = np.load("features/labels.npy")

# ─────────────────────────────────────────────────────────────────────────
# Costruzione tf.data.Dataset pronti per il training
# ─────────────────────────────────────────────────────────────────────────

BATCH_SIZE  = 32
AUTOTUNE    = tf.data.AUTOTUNE

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,   # sklearn vuole numpy
    test_size=0.2,     # 20% validation
    stratify=y_train,  # mantiene distribuzione classi
    random_state=42
)

# Converti y in tensor
y_train_final = tf.convert_to_tensor(y_train_final)
y_val         = tf.convert_to_tensor(y_val)

# Converti X in float32 (range approssimativo [-1, 1] dopo rescaling)
# Il modello riceve (49, 32, 1) – aggiunta dim canale per Conv2D.
def make_dataset(X: np.ndarray, y: tf.Tensor, shuffle: bool) -> tf.data.Dataset:
    # float32 normalizzato in [-1, 1] per facilitare il training
    X_f = (X.astype(np.float32) / 128.0).reshape(len(X), -1)  # (N, 1568)
    ds  = tf.data.Dataset.from_tensor_slices((X_f, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train_final, y_train_final, shuffle=True)
val_ds   = make_dataset(X_val,   y_val,   shuffle=False)
test_ds  = make_dataset(X_test,  y_test,  shuffle=False)

print("\ntrain_ds:", train_ds)
print("val_ds  :", val_ds)
print("test_ds :", test_ds)


train_ds: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 1568), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.int64, name=None))>
val_ds  : <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 1568), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.int64, name=None))>
test_ds : <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 1568), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.int64, name=None))>


E0000 00:00:1776511735.622749    7999 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1776511735.641738    7999 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


# Load tflite model

In [3]:
from ai_edge_litert.interpreter import Interpreter
import numpy as np

interpreter = Interpreter(model_path="models/MLP.tflite")
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

out_scale, out_zero_point = output_details[0]['quantization']
in_scale,  in_zero_point  = input_details[0]['quantization']

correct = 0
total   = len(X_test)

for i in range(total):
    input_data = X_test[i].flatten().astype(np.float32)

    if input_details[0]['dtype'] == np.int8:
        input_data = (input_data / in_scale + in_zero_point).round().astype(np.int8)

    input_data = np.expand_dims(input_data, axis=0)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])

    if out_scale > 0:
        output_data = (output_data.astype(np.float32) - out_zero_point) * out_scale

    predicted = np.argmax(output_data)

    # Gestisce label scalare (es. 3) o one-hot (es. [0,0,1,0])
    y = y_test[i]
    expected = int(np.argmax(y)) if np.ndim(y) > 0 and np.size(y) > 1 else int(y.item())

    if predicted == expected:
        correct += 1

accuracy = correct / total * 100
print(f"Accuracy sul test set: {correct}/{total} ({accuracy:.2f}%)")

Accuracy sul test set: 461/641 (71.92%)


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
